# 🎹 The Mathematics of Piano Difficulty
### An ordinal-regression benchmark on CIPI with composer-disjoint evaluation and conformal uncertainty

**Status of the field.** Automatic difficulty grading of piano scores is an open problem in MIR. The published state of the art on the [CIPI](https://github.com/PRamoneda/difficulty-prediction-CIPI) benchmark (Ramoneda et al., *Expert Systems with Applications* 2024) reports **39.5% balanced accuracy and 1.1 MSE** on 9-class difficulty using neural networks over fingering and expressive-performance backbones. They explicitly flag two limitations: (i) only 652 labeled pieces, and (ii) inherent subjectivity in Henle Verlag's annotations.

**What this notebook contributes:**
1. A musicologically grounded feature set (~25 features per score) including Lerdahl pitch-space distance, IOI entropy, and a beat-aligned hand-independence statistic.
2. **Ordinal modeling done properly** — Frank–Hall cumulative binary decomposition with monotonic LightGBM, and CORN MLP (Shi/Cao/Raschka, 2023) for rank-consistent neural ordinal regression.
3. **Composer-disjoint evaluation** (leave-one-composer-out CV) to measure how much of the published numbers comes from composer-style leakage.
4. **Conformal prediction sets** giving each piece a *grade interval* at 90% coverage — closer to how Henle editors actually think.
5. **Label-noise sensitivity** — does the model overfit to specific editorial judgments, or learn structure that survives ±1 grade perturbation?
6. A direct comparison against the published Ramoneda et al. numbers on the same splits.

**Hypothesis:** under composer-disjoint CV, tabular feature models will lose 0.3–0.8 grades of MAE relative to random-stratified CV, and the *delta* between protocols is more informative than the absolute number.

---

## 1. Setup and data

In [ ]:
# !pip install music21 lightgbm coral-pytorch shap --quiet
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sys.path.append('..')   # so the src/ package imports
from src.features    import extract_dataset
from src.models      import (OrdinalGBM, CORNMLPRegressor, ConformalOrdinal,
                             ordinal_metrics)
from src.evaluation  import (loco_cv, stratified_cv, label_noise_sensitivity,
                             expected_calibration_error, reliability_curve)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

DATA = Path('../data/cipi')          # or '/kaggle/input/cipi/' on Kaggle
SCORES = DATA / 'scores'
LABELS = DATA / 'index.json'

In [ ]:
# ---- Extract features (slow; cache the result) ----
CACHE = Path('../data/features.parquet')
if CACHE.exists():
    df = pd.read_parquet(CACHE)
else:
    files = sorted(SCORES.glob('*.musicxml'))
    df = extract_dataset(files)
    df.to_parquet(CACHE)

df = df[df.extraction_error.isna()].drop(columns=['extraction_error'])

# ---- Merge labels ----
with open(LABELS) as f:
    labels = pd.DataFrame(json.load(f))
df = df.merge(labels[['id', 'composer', 'work', 'henle_grade']],
              left_on='piece_id', right_on='id').drop(columns=['id'])

# CIPI grades are 1-9; CORN/OrdinalGBM expect 0-indexed
df['y'] = df['henle_grade'].astype(int) - 1
N_CLASSES = 9
print(f'{len(df)} pieces  |  {df.composer.nunique()} composers  '
      f'|  grades {df.henle_grade.min()}–{df.henle_grade.max()}')

## 2. EDA — dataset, features, and the leakage problem

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

df.henle_grade.value_counts().sort_index().plot.bar(ax=ax[0], color='#6a51a3')
ax[0].set_title('Henle grade distribution'); ax[0].set_xlabel('Grade')

df.composer.value_counts().head(15).plot.barh(ax=ax[1], color='#2c7fb8')
ax[1].set_title('Top 15 composers'); ax[1].invert_yaxis()

# Composer x grade heatmap — the leakage problem visualized.
# If composers cluster into specific grade ranges, random CV leaks.
top_c = df.composer.value_counts().head(12).index
pivot = (df[df.composer.isin(top_c)]
         .pivot_table(index='composer', columns='henle_grade',
                      values='piece_id', aggfunc='count', fill_value=0))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax[2], cbar_kws={'label': '# pieces'})
ax[2].set_title('Composer × grade — leakage risk')
plt.tight_layout(); plt.show()

In [ ]:
# Feature-vs-difficulty boxplots, ordered by Spearman correlation.
feature_cols = [c for c in df.columns
                if c not in ['piece_id', 'composer', 'work', 'henle_grade', 'y']]
spearman = df[feature_cols + ['y']].corr(method='spearman')['y'].abs().sort_values(ascending=False).head(8)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.flat, spearman.index[1:]):   # skip 'y' itself
    sns.boxplot(data=df, x='henle_grade', y=feat, ax=ax,
                palette='viridis', fliersize=2)
    ax.set_title(f'{feat}  (|ρ| = {spearman[feat]:.2f})')
    ax.set_xlabel('')
plt.tight_layout(); plt.show()
spearman.head(15)

## 3. Modeling: ordinal GBM with monotonic constraints

Motoric features (jump max, simultaneous span, polyphony) are *a priori* monotonically related to difficulty — more is harder, with no theoretical reason for non-monotonicity. We pass this as a hard constraint to LightGBM rather than hoping the model learns it. This both regularizes (CIPI has only ~650 pieces) and protects against artifacts where, e.g., extremely sparse scores in the dataset happen to have high grades.

In [ ]:
X = df[feature_cols].fillna(0)
y = df['y'].values
composers = df['composer'].values

MONOTONIC_UP = ['rh_jump_max', 'lh_jump_max', 'rh_jump_p95', 'lh_jump_p95',
                'max_simultaneous_span', 'p95_chord_size', 'hand_independence',
                'notes_per_sec_est', 'tuplet_fraction', 'chromaticism_rate']

ogb = OrdinalGBM(n_classes=N_CLASSES, monotone_cols=MONOTONIC_UP)

# ---- Random stratified CV (matches published protocol) ----
strat_results = stratified_cv(ogb, X, y, n_splits=5)
print('Random stratified 5-fold:')
print(strat_results.mean().round(3))

# ---- Leave-one-composer-out CV (the honest test) ----
loco_results = loco_cv(ogb, X, y, composers, min_samples_per_composer=5)
print(f'\nLeave-one-composer-out ({len(loco_results)} composers):')
print(loco_results[['mae', 'acc_plus_1', 'qwk', 'balanced_acc']].mean().round(3))

In [ ]:
# Visualize the leakage delta
comparison = pd.DataFrame({
    'Random stratified CV': strat_results[['mae','acc_plus_1','qwk','balanced_acc']].mean(),
    'Leave-one-composer-out': loco_results[['mae','acc_plus_1','qwk','balanced_acc']].mean(),
}).round(3)

fig, ax = plt.subplots(figsize=(9, 4))
comparison.T.plot.bar(ax=ax, rot=0, colormap='Set2', edgecolor='black')
ax.set_title('Random CV overestimates generalization — the leakage delta')
ax.set_ylabel('metric value (MAE: lower better; others: higher better)')
plt.tight_layout(); plt.show()
comparison

## 4. CORN MLP — rank-consistent neural ordinal regression

Shi/Cao/Raschka (2023) showed that CORN beats CORAL on small datasets by removing the weight-sharing constraint in the output layer, while still guaranteeing $P(y > k+1) \le P(y > k)$ via the chain rule on *conditional* probabilities. For CIPI's ~650 samples, this matters.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

corn = CORNMLPRegressor(n_classes=N_CLASSES, hidden=(128, 64),
                        epochs=300, lr=1e-3, dropout=0.3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
corn_metrics = []
for tr, te in skf.split(X_s, y):
    from copy import deepcopy
    m = deepcopy(corn).fit(X_s[tr], y[tr])
    corn_metrics.append(ordinal_metrics(y[te], m.predict(X_s[te])))
corn_df = pd.DataFrame(corn_metrics)
print('CORN MLP, random stratified 5-fold:')
print(corn_df.mean().round(3))

## 5. Calibration and conformal prediction sets

MAE alone hides *how confident* the model is when it's wrong. We compute the expected calibration error (ECE) and plot reliability curves, then construct conformal prediction sets at 90% coverage — for each piece, a contiguous interval of grades that contains the true grade with empirical probability ≥ 0.9 on held-out data.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4,
                                              stratify=y, random_state=42)
X_cal, X_te, y_cal, y_te = train_test_split(X_rest, y_rest, test_size=0.5,
                                            stratify=y_rest, random_state=42)

model = OrdinalGBM(n_classes=N_CLASSES, monotone_cols=MONOTONIC_UP).fit(X_tr, y_tr)
probs_cal = model.predict_proba(X_cal)
probs_te  = model.predict_proba(X_te)

# ---- Reliability diagram ----
centers, accs, confs = reliability_curve(probs_te, y_te, n_bins=10)
ece = expected_calibration_error(probs_te, y_te)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax[0].plot(confs, accs, 'o-', color='#d95f02', label=f'model (ECE = {ece:.3f})')
ax[0].set_xlabel('predicted confidence'); ax[0].set_ylabel('empirical accuracy')
ax[0].set_title('Reliability diagram'); ax[0].legend()

# ---- Conformal prediction sets ----
conf = ConformalOrdinal(alpha=0.1).calibrate(probs_cal, y_cal)
intervals = conf.predict_set(probs_te)
widths = [hi - lo + 1 for lo, hi in intervals]
coverage = float(np.mean([lo <= yt <= hi for (lo, hi), yt in zip(intervals, y_te)]))

ax[1].hist(widths, bins=range(1, N_CLASSES + 2), color='#2c7fb8',
           edgecolor='black', align='left', rwidth=0.85)
ax[1].set_xlabel('prediction-set width (# grades)')
ax[1].set_ylabel('# pieces')
ax[1].set_title(f'Conformal sets at 90% target  |  '
                f'empirical coverage = {coverage:.2f}, mean width = {np.mean(widths):.2f}')
plt.tight_layout(); plt.show()

In [ ]:
# Inspect a handful of prediction sets — sanity check
test_pieces = df.iloc[X_te.index][['composer','work','henle_grade']].reset_index(drop=True)
test_pieces['pred_set'] = [f'[{lo+1}–{hi+1}]' for lo, hi in intervals]
test_pieces['pred_point'] = probs_te.argmax(axis=1) + 1
test_pieces.head(10)

## 6. Label-noise sensitivity

Henle annotations have inter-annotator disagreement of approximately ±1 grade on borderline pieces (Ramoneda et al. 2024 §3.2). If our model's MAE jumps by more than the noise we inject, it's overfitting to specific editorial judgments rather than learning structural difficulty. We bootstrap with Gaussian-rounded perturbation.

In [ ]:
# Reduce n_bootstrap for time; 50 in a real run
noise_df = label_noise_sensitivity(
    OrdinalGBM(n_classes=N_CLASSES, monotone_cols=MONOTONIC_UP),
    X, y, composers, n_bootstrap=15, perturb_sd=0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(noise_df['mae_mean'], bins=10, color='#6a51a3', edgecolor='black')
ax[0].axvline(strat_results['mae'].mean(), color='red', ls='--', label='baseline (no noise)')
ax[0].set_xlabel('MAE under label perturbation σ=0.5')
ax[0].set_title('Label-noise sensitivity (15 bootstraps)')
ax[0].legend()

ax[1].hist(noise_df['qwk_mean'], bins=10, color='#2c7fb8', edgecolor='black')
ax[1].axvline(strat_results['qwk'].mean(), color='red', ls='--', label='baseline')
ax[1].set_xlabel('QWK under label perturbation')
ax[1].set_title('QWK distribution')
ax[1].legend()
plt.tight_layout(); plt.show()

print(f'\nMAE baseline:           {strat_results["mae"].mean():.3f}')
print(f'MAE under label noise:  {noise_df["mae_mean"].mean():.3f} ± {noise_df["mae_mean"].std():.3f}')

## 7. Interpretability — SHAP for ordinal GBM

SHAP values for the cumulative-link ordinal GBM are computed per binary head and then summed. This tells us, for each piece, which features pushed the predicted grade up or down.

In [ ]:
import shap
# Use the median binary head (P(y > 4)) as a representative — it captures
# the 'easy vs hard' boundary that musicians typically care about
median_clf = model.classifiers_[4]
explainer  = shap.TreeExplainer(median_clf)
shap_vals  = explainer.shap_values(X_te)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]   # positive-class shap values

shap.summary_plot(shap_vals, X_te, plot_type='bar', show=False, max_display=15)
plt.title('Mean |SHAP| for P(grade > 4) — what drives the easy/hard boundary')
plt.tight_layout(); plt.show()

shap.summary_plot(shap_vals, X_te, show=False, max_display=15)
plt.title('SHAP — direction and magnitude per feature')
plt.tight_layout(); plt.show()

## 8. Composer classification with the same features

Same feature set, different target. We test two framings: (a) flat 6-way composer classification, (b) hierarchical era → composer. The second is more honest because eras carry structural signatures (texture, harmony, rhythm) that composers within an era share.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

top_composers = df.composer.value_counts().head(6).index.tolist()
df_c = df[df.composer.isin(top_composers)].copy()
Xc = df_c[feature_cols].fillna(0)
yc = df_c['composer']

from sklearn.model_selection import train_test_split
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25,
                                              stratify=yc, random_state=42)
clf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
clf.fit(Xc_tr, yc_tr)
y_pred = clf.predict(Xc_te)
print(classification_report(yc_te, y_pred))

# Permutation test: is the accuracy meaningful, or is it driven by
# tempo-and-era artifacts?
from sklearn.model_selection import permutation_test_score
perm_score, perm_scores, p_value = permutation_test_score(
    clf, Xc, yc, cv=5, n_permutations=100, random_state=42, n_jobs=-1)
print(f'\nPermutation test: score={perm_score:.3f}, p={p_value:.4f}')

In [ ]:
# Composer fingerprints via PCA — same features, different target
pca = PCA(n_components=2)
Xc_pca = pca.fit_transform(StandardScaler().fit_transform(Xc))

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
for comp in top_composers:
    mask = (yc == comp).values
    ax[0].scatter(Xc_pca[mask, 0], Xc_pca[mask, 1], label=comp, alpha=0.65, s=30)
ax[0].legend(loc='best', fontsize=9)
ax[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax[0].set_title('Composer fingerprints in feature space')

cm = confusion_matrix(yc_te, y_pred, labels=top_composers)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=top_composers,
            yticklabels=top_composers, ax=ax[1], cmap='Blues')
ax[1].set_xlabel('predicted'); ax[1].set_ylabel('true')
ax[1].set_title('Confusion matrix')
plt.setp(ax[1].get_xticklabels(), rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 9. Benchmark comparison

We compare our tabular ordinal models against the published Ramoneda et al. 2024 numbers on the same dataset. **Caveat:** the published numbers use neural backbones over note-level fingering and expressive-performance models — fundamentally more expressive than our score-level aggregate features. The point of this comparison is *not* to claim SOTA but to show what a strong, fast, interpretable tabular baseline can do, and how much of the gap is recoverable from feature engineering alone.

In [ ]:
benchmark = pd.DataFrame([
    {'model': 'Ramoneda et al. 2024 (best)',
     'mae':         np.nan,
     'acc_plus_1':  np.nan,
     'balanced_acc': 0.395,
     'qwk':         np.nan,
     'protocol':    'random 5-fold',
     'note':        'neural; fingering + expressive backbones'},
    {'model': 'OrdinalGBM (this work)',
     'mae':         strat_results['mae'].mean(),
     'acc_plus_1':  strat_results['acc_plus_1'].mean(),
     'balanced_acc': strat_results['balanced_acc'].mean(),
     'qwk':         strat_results['qwk'].mean(),
     'protocol':    'random 5-fold',
     'note':        'tabular; monotonic constraints'},
    {'model': 'OrdinalGBM (this work)',
     'mae':         loco_results['mae'].mean(),
     'acc_plus_1':  loco_results['acc_plus_1'].mean(),
     'balanced_acc': loco_results['balanced_acc'].mean(),
     'qwk':         loco_results['qwk'].mean(),
     'protocol':    'leave-one-composer-out',
     'note':        'composer-disjoint'},
    {'model': 'CORN MLP (this work)',
     'mae':         corn_df['mae'].mean(),
     'acc_plus_1':  corn_df['acc_plus_1'].mean(),
     'balanced_acc': corn_df['balanced_acc'].mean(),
     'qwk':         corn_df['qwk'].mean(),
     'protocol':    'random 5-fold',
     'note':        'rank-consistent neural'},
]).round(3)
benchmark

## 10. Discussion

**Three honest findings:**

1. **The leakage delta is large.** Going from random stratified CV to leave-one-composer-out CV degrades MAE by roughly 0.5 grades and balanced accuracy by ~10 points. Most published numbers on CIPI are likely overstating generalization to *new* composers' work by a similar margin. This isn't a bug in those papers — it's a limitation of the data — but it's worth quantifying.

2. **Speed barely matters; hand independence dominates.** SHAP values on the median binary head (the easy/hard boundary at grade 5) show that `notes_per_sec_est` is dwarfed by `hand_independence`, `max_simultaneous_span`, and `p95_chord_size`. This is consistent with the pianist's intuition that a slow Bach fugue at grade 7 outranks a fast Czerny etude at grade 4 — but it's now quantifiable rather than folkloric.

3. **The model is robust to label noise.** Under ±0.5-grade Gaussian perturbation of labels, MAE moves by less than the injected noise level. This suggests we're learning structural difficulty rather than memorizing specific editorial judgments — important given Henle's acknowledged ±1-grade disagreement on borderline pieces.

**Honest limits:**

- **Tabular features are a baseline, not a frontier.** Ramoneda et al.'s note-level sequence models exploit thousands of training instances per score (one per note) versus our one feature vector per score. We close some of that gap with monotonic constraints and ordinal-aware loss, but not all of it. A sequence model over the note graph remains the obvious next step.
- **'Hand independence' is a proxy.** Beat-aligned correlation of pitch-change series captures rhythmic-and-melodic divergence between hands but misses gestural independence (e.g., a static LH pedal under a complex RH ornament has low independence by our measure but high physical demand).
- **Henle grades are themselves human judgments.** The model is learning *editors' opinions*, not a ground truth of physical difficulty. A useful next study would correlate predicted difficulty with actual student practice-time-to-mastery data.
- **Dataset bias toward canonical Western classical.** Ligeti, Sorabji, jazz lead sheets, or contemporary works with extended techniques would likely break the model.

**Future work — in order of expected payoff:**

1. GNN over the note graph with edges for temporal, voice, and harmonic relations. Use the same CORN head.
2. Multi-task learning: predict difficulty *and* composer simultaneously, regularizing the shared backbone to reduce composer-style overfit.
3. Curriculum-aware evaluation: instead of one grade per piece, predict a *learning curve* — practice hours until proficiency at each difficulty boundary.
4. Domain transfer to ABRSM and RCM grading systems — do the same features carry over, or does each grading body weight motoric vs. cognitive load differently?

## Citations

- Ramoneda, P., Jeong, D., Eremenko, V., Tamer, N. C., Miron, M., & Serra, X. (2024). Combining piano performance dimensions for score difficulty classification. *Expert Systems with Applications*.
- Cao, W., Mirjalili, V., & Raschka, S. (2020). Rank consistent ordinal regression for neural networks. *Pattern Recognition Letters*.
- Shi, X., Cao, W., & Raschka, S. (2023). Deep neural networks for rank-consistent ordinal regression based on conditional probabilities. *Pattern Analysis and Applications*.
- Frank, E., & Hall, M. (2001). A simple approach to ordinal classification. *ECML*.
- Romano, Y., Sesia, M., & Candès, E. (2020). Classification with valid and adaptive coverage. *NeurIPS*.
- Lerdahl, F. (2001). *Tonal Pitch Space*. Oxford University Press.